# Chapter 18: Reinforcement Learning

## Global Imports

In [1]:
import gym
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

# Check versions
print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
try:
    print(f"Gym Version: {gym.__version__}")
except:
    print("Gym not installed or version not found.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


TensorFlow Version: 2.19.0
Keras Version: 3.10.0
Gym Version: 0.25.2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Learning to Optimize Rewards
Reinforcement Learning (RL) is a subfield of Machine Learning where an Agent learns to make decisions by performing actions in an Environment and observing the results.

- Key Components:Agent:
- The learner (e.g., a robot, a game bot).
- Environment: The world the agent interacts with (e.g., the game level, the laws of physics).
- Action ($a_t$): What the agent does at time step $t$.
- Observation ($O_t$): What the agent perceives (state of the environment).
- Reward ($R_t$): A scalar feedback signal indicating how good or bad the action was.

The Goal: The agent's objective is to maximize the expected cumulative reward over time, often called the return. Unlike supervised learning, the agent is not told what to do; it must discover the best actions through trial and error.

<p align="left"><img src="../fig/figure18.1.png" width="45%"></p>

## Introduction to OpenAI Gym
OpenAI Gym (now maintained as Gymnasium) is a toolkit for developing and comparing RL algorithms. It provides a standard API for environments.

We will use the CartPole environment. The goal is to balance a pole on a moving cart.
- Observations: [Cart Position, Cart Velocity, Pole Angle, Pole Angular Velocity].
- Actions: 0 (Push Left) or 1 (Push Right).
- Reward: +1 for every time step the pole remains upright.

<p align="left"><img src="../fig/figure18.4.png" width="45%"></p>

### Code Example: Running a Basic CartPole Environment
We initialize the environment, reset it to get the first observation, and run a loop taking random actions.

In [6]:
import gym
import numpy as np

# --- PERBAIKAN: Tambahkan baris ini untuk mengatasi error numpy ---
np.bool8 = np.bool_
# ----------------------------------------------------------------

env = gym.make("CartPole-v1", render_mode="rgb_array")

# Reset environment
try:
    obs = env.reset(seed=42)
except TypeError:
    env.seed(42)
    obs = env.reset()

print(f"Initial Observation: {obs}")
print(f"Action Space: {env.action_space}")

total_reward = 0
for _ in range(5):
    action = env.action_space.sample()

    # Menggunakan API lama (4 return values)
    obs, reward, done, info = env.step(action)

    total_reward += reward
    print(f"Action: {action}, New Obs: {obs}, Reward: {reward}")

    if done:
        break

env.close()
print(f"Total Reward gathered: {total_reward}")

Initial Observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
Action Space: Discrete(2)
Action: 0, New Obs: [ 0.02727336 -0.20172954  0.03625453  0.32351476], Reward: 1.0
Action: 0, New Obs: [ 0.02323877 -0.39734846  0.04272482  0.62740684], Reward: 1.0
Action: 1, New Obs: [ 0.0152918  -0.20284806  0.05527296  0.34847975], Reward: 1.0
Action: 0, New Obs: [ 0.01123484 -0.39871082  0.06224256  0.65806717], Reward: 1.0
Action: 0, New Obs: [ 0.00326062 -0.5946414   0.0754039   0.9696814 ], Reward: 1.0
Total Reward gathered: 5.0


Explanation: The agent survived for 5 steps, earning +1.0 reward each step. The observation vector changes as the physics simulation updates.

## Policies
A Policy ($\pi$) is the strategy the agent uses to determine the next action based on the current state.
- Deterministic Policy: $a = \pi(s)$. Always returns the same action for a given state.
- Stochastic Policy: $\pi(a|s) = P(A_t=a | S_t=s)$. Returns a probability distribution over actions. This allows the agent to explore.

<p align="left"><img src="../fig/figure18.2.png" width="45%"></p>

### Neural Network Policy
We can use a Neural Network to learn the policy. It takes the observation as input and outputs the probability of taking a specific action (e.g., "Right").

### Code Example: Creating a Policy Network
This is a simple binary classification network (but trained via RL, not labels).

In [7]:
model = keras.models.Sequential([
    keras.layers.Dense(5, activation="elu", input_shape=[4]), # Input: 4 obs features
    keras.layers.Dense(1, activation="sigmoid") # Output: Prob of moving LEFT (0)
])

# Test the model with a dummy observation
dummy_obs = tf.constant([[0.0, 0.1, -0.05, 0.0]]) # Batch size 1
prob_left = model(dummy_obs)
action = int(tf.random.uniform([1, 1]) > prob_left) # Sample action

print(f"Probability of Left: {prob_left.numpy()[0,0]:.4f}")
print(f"Selected Action: {action} (0=Left, 1=Right)")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Probability of Left: 0.5000
Selected Action: 1 (0=Left, 1=Right)


/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py:312: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return int(self._numpy())


Explanation: The untrained network outputs a probability near 0.5. We sample from this probability to decide the action.

<p align="left"><img src="../fig/figure18.5.png" width="45%"></p>

## The Credit Assignment Problem & Discounted Rewards
In RL, a reward often comes after many actions. If an agent balances the pole for 100 steps and then drops it, which of the 100 actions were good and which caused the fall? This is the Credit Assignment Problem.

To solve this, we use Discounted Rewards. We assume that immediate rewards are more valuable than future rewards. The Return ($G_t$) is the sum of discounted future rewards:$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

- $\gamma$ (gamma): The discount factor (typically 0.95 to 0.99).
- If $\gamma$ is close to 0, the agent is short-sighted.
- If $\gamma$ is close to 1, the agent cares about the far future.

### Code Example: Calculating Discounted Rewards

In [8]:
def discount_rewards(rewards, gamma):
    discounted = np.array(rewards)
    for step in range(len(rewards) - 2, -1, -1):
        discounted[step] += discounted[step + 1] * gamma
    return discounted

raw_rewards = [10.0, 0.0, -50.0] # Win, Wait, Crash
gamma = 0.8
disc_rewards = discount_rewards(raw_rewards, gamma)

print(f"Raw Rewards: {raw_rewards}")
print(f"Discounted Rewards: {disc_rewards}")

Raw Rewards: [10.0, 0.0, -50.0]
Discounted Rewards: [-22. -40. -50.]


Explanation: The last action caused a crash (-50). The middle action led to the crash, so it inherits a negative value (-40). Even the first action, which seemed good (+10), is penalized because it led to a bad future (-22).

## Policy Gradients (PG)
The REINFORCE algorithm acts by tweaking the neural network parameters ($\theta$) to increase the probability of actions that resulted in high rewards.

Logic:

1. Play several episodes using the current policy.
2. Calculate the discounted rewards (returns) for every step.
3. Normalize rewards (subtract mean, divide by std) to judge if an action was "better than average".
4. Compute gradients to make the chosen actions more likely if the reward was positive, or less likely if negative.
5. Apply gradients to weights.

### Code Example: One Training Step of Policy Gradient

In [10]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

def play_one_step(env, obs, model, loss_fn):
    with tf.GradientTape() as tape:
        # 1. Predict
        left_proba = model(obs[np.newaxis]) # Obs shape needs to be (1, 4)

        # 2. Sample Action
        action = (tf.random.uniform([1, 1]) > left_proba)
        y_target = tf.constant([[1.]]) - tf.cast(action, tf.float32)

        # 3. Calculate Loss
        loss = tf.reduce_mean(loss_fn(y_target, left_proba))

    grads = tape.gradient(loss, model.trainable_variables)

    # PERBAIKAN 1: API Lama mengembalikan 4 nilai (tanpa truncated)
    obs, reward, done, info = env.step(int(action[0, 0].numpy()))

    return obs, reward, done, grads

# Simulated execution
loss_fn = keras.losses.BinaryCrossentropy()

# PERBAIKAN 2: API Lama reset() hanya mengembalikan obs (1 nilai)
try:
    obs = env.reset(seed=42)
except TypeError:
    # Fallback untuk gym versi sangat lama
    env.seed(42)
    obs = env.reset()

obs, reward, done, grads = play_one_step(env, obs, model, loss_fn)

print(f"Action taken resulted in Reward: {reward}")
print(f"Gradient shape for first layer: {grads[0].shape}")

Action taken resulted in Reward: 1.0
Gradient shape for first layer: (4, 5)


Explanation: We computed the gradients that would make the selected action more probable. We will multiply these gradients by the actual reward (advantage) later before applying them.

## Markov Decision Processes (MDP) & Q-Learning
To train more robust agents, we often use Value-Based methods.
An MDP is defined by $(S, A, P, R, \gamma)$.
The Bellman Optimality Equation states that the optimal value of a state $V^*(s)$ is the max reward we can get plus the discounted value of the next state:

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a) [R(s,a,s') + \gamma V^*(s')]$$

Q-Learning focuses on the Quality (Q-Value) of a state-action pair, $Q(s, a)$.
$Q(s, a)$: "How much total reward will I get if I am in state $s$, take action $a$, and then play optimally?"

The update rule (Temporal Difference) is:

$$Q(s, a) \leftarrow (1-\alpha)Q(s, a) + \alpha \left( r + \gamma \max_{a'} Q(s', a') \right)$$


<p align="left"><img src="../fig/figure18.7.png" width="45%"></p>
<p align="left"><img src="../fig/figure18.8.png" width="45%"></p>

## Deep Q-Learning (DQN)
For complex environments (like video games or continuous states), we cannot store a Q-table for every state. Instead, we approximate the Q-function using a Neural Network: $Q_{\theta}(s, a)$. This is a Deep Q-Network (DQN).

Critical Components for Stability:

1. Experience Replay: We store experiences $(s, a, r, s', done)$ in a buffer and sample a random batch for training. This breaks correlations between consecutive steps (which usually confuse Neural Nets).
2. Target Network: We use two networks.
- Online Model: learns and predicts actions.
- Target Model: calculates the target Q-values. Its weights are frozen and updated only occasionally (e.g., every 1000 steps). This prevents the "moving target" problem where the network chases its own tail.

### Code Example: Implementing a DQN Agent

In [12]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

input_shape = [4] # CartPole state size
n_outputs = 2 # Left or Right

# 1. Create the Online Model
model = keras.models.Sequential([
    keras.layers.Input(shape=input_shape),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(n_outputs)
])

# 2. Target Model
target = keras.models.clone_model(model)
target.set_weights(model.get_weights())

# 3. Epsilon-Greedy Policy
def epsilon_greedy_policy(state, epsilon=0):
    if np.random.rand() < epsilon:
        return np.random.randint(2)
    else:
        Q_values = model.predict(state[np.newaxis], verbose=0)
        return np.argmax(Q_values[0])

# 4. Training Step
def training_step(batch_size, discount_rate, optimizer, replay_buffer):
    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

    # Calculate Target Q-Values
    next_Q_values = target.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)

    # Agar dimensi cocok saat penjumlahan (batch, 1) + (batch, 1)
    # Kita expand max_next_Q_values dari (32,) menjadi (32, 1)
    max_next_Q_values = max_next_Q_values.reshape(-1, 1)

    target_Q_values = (rewards + (1 - dones) * discount_rate * max_next_Q_values)

    mask = tf.one_hot(actions, n_outputs)

    loss_fn = keras.losses.MeanSquaredError()

    with tf.GradientTape() as tape:
        all_Q_values = model(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)

        loss = loss_fn(target_Q_values, Q_values)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

# Mock Buffer
class MockReplayBuffer:
    def sample(self, batch_size):
        s = np.random.randn(batch_size, 4).astype(np.float32)
        a = np.random.randint(0, 2, size=batch_size)
        r = np.random.rand(batch_size, 1).astype(np.float32)
        ns = np.random.randn(batch_size, 4).astype(np.float32)
        d = np.random.randint(0, 2, size=(batch_size, 1)).astype(np.float32)
        return s, a, r, ns, d

optimizer = keras.optimizers.Adam(learning_rate=1e-3)
loss = training_step(32, 0.95, optimizer, MockReplayBuffer())

print(f"DQN Training Step Loss: {loss.numpy():.4f}")

DQN Training Step Loss: 0.5190


Explanation: The loss represents the difference between the Q-value predicted by our model and the "True" Q-value estimated using the immediate reward plus the Target Model's prediction. Minimizing this loss allows the agent to learn accurate value estimates.

## DQN Variants
The book discusses several improvements to stabilize and speed up DQN:

1. Double DQN: In standard DQN, the max operator uses the same values to select and evaluate an action, leading to overestimation. Double DQN uses the Online model to select the best action and the Target model to estimate its value.
2. Dueling DQN: Splits the network into two streams:
- Value Stream $V(s)$: How good is the state itself?
- Advantage Stream $A(s, a)$: How much better is this action compared to others?
- Final Q-Value: $Q(s, a) = V(s) + A(s, a)$.
- This helps because often the value of a state matters more than the specific action choice.

The chapter concludes by mentioning libraries like TF-Agents and Ray RLLib, which provide optimized implementations of these algorithms for production use.